# Activity 5 Concepts: yfinance and Smoothing

Activity 5 adds a second, live data source to your app: daily stock prices pulled with
`yfinance`. It also adds a rolling average the user controls with a slider. Both of these have a
trap that is much easier to see with printed output than inside a Streamlit script: `yfinance`
returns a shape of data you probably do not expect by default, and a rolling average has a cost
that is easy to forget once it is hidden behind a slider.

This notebook shows you both, on their own, before you write `activity_5_demand_explorer.py`.

Import pandas and `yfinance`. Expect no output, the cell just runs.

In [ ]:
import pandas as pd
import yfinance as yf

Download a small window of AAPL data with `yf.download`, using only the arguments you would
reach for first: a ticker, a start date, and an end date. Look at `frame.columns`. Expect each
column name to be a TUPLE, like `('Close', 'AAPL')`, not a plain string like `'Close'`. This is a
`MultiIndex`: yfinance labels every column with both the field (Close, High, Low, Open, Volume)
and the ticker, in case you ever download more than one ticker at once.

In [ ]:
frame = yf.download("AAPL", start="2024-01-02", end="2024-01-12", progress=False)
frame.columns

Now select the `"Close"` column the way you normally would, and check its type. Expect a
`DataFrame`, not a `Series`. With MultiIndex columns, `frame["Close"]` matches every column whose
top level is `"Close"`, and with one ticker downloaded that is still one column, but pandas keeps
it as a one-column DataFrame rather than collapsing it to a Series.

In [ ]:
print(type(frame["Close"]))
frame["Close"].head(3)

A one-column DataFrame looks almost like a Series when you print it, but it is not one. Any
call written for a Series, `.rolling()`, `.mean()`, `.iloc[-1]`, either raises an error or quietly
does the wrong thing on a DataFrame. This is the trap: the bug does not show up here, it shows up
later, in code that assumed `frame["Close"]` was a Series.

Download the same data again, this time passing `auto_adjust=True` and
`multi_level_index=False` together. Look at `frame2.columns`. Expect flat strings now,
`'Close'`, `'High'`, `'Low'`, `'Open'`, `'Volume'`, no tuples, no MultiIndex. Both keyword
arguments are required together to get this shape.

In [ ]:
frame2 = yf.download(
    "AAPL", start="2024-01-02", end="2024-01-12", progress=False,
    auto_adjust=True, multi_level_index=False,
)
frame2.columns

Select `"Close"` from the flat-column version and check its type again. Expect a `Series`
this time. This is the shape `load_prices` in the app needs to return, so `.rolling()` and
`.mean()` work on it without surprises.

In [ ]:
print(type(frame2["Close"]))
frame2["Close"].head(3)

Now the second concept: a rolling window. Before touching the taxi data, build a tiny Series
by hand, 10 numbers you chose yourself, small enough to check the arithmetic without pandas.

In [ ]:
tiny = pd.Series([10, 12, 9, 14, 20, 18, 15, 13, 11, 16])
tiny

Call `.rolling(3).mean()` on it. Expect the first two positions to be `NaN`: a 3-value rolling
average needs 3 values to average, and positions 0 and 1 do not have two prior values yet.
Position 2 is the first real average: `(10 + 12 + 9) / 3 = 10.33`. Check that by hand against the
printed value, then check position 3 the same way: `(12 + 9 + 14) / 3 = 11.67`.

In [ ]:
tiny.rolling(3).mean()

With the arithmetic confirmed on data small enough to check by eye, apply the same idea to the
real series. Reload the taxi data and resample it to daily totals, exactly as in the Activity 4
notebook.

In [ ]:
raw = pd.read_csv("data/nyc_taxi.csv", parse_dates=["timestamp"], index_col="timestamp")
daily = raw["value"].resample("D").sum()
daily.head()

Take a 7-day rolling average. Expect the first 6 rows to be `NaN`, for the same reason as the
tiny series: a 7-value window needs 7 prior values before it can produce its first real number.

In [ ]:
rolling_7 = daily.rolling(7).mean()
rolling_7.head(8)

Take a 30-day rolling average of the same series, and plot the raw daily total against both
rolling averages. Expect the raw line to be jagged, the 7-day line to be visibly smoother, and the
30-day line smoother still, a wide, slow-moving curve.

In [ ]:
import matplotlib.pyplot as plt

rolling_30 = daily.rolling(30).mean()
pd.DataFrame({"raw": daily, "7-day average": rolling_7, "30-day average": rolling_30}).plot(
    figsize=(12, 5),
    title="Daily rides, raw vs 7-day vs 30-day rolling average",
)
plt.show()

Smoothing is not free. A larger window does not just remove noise, it can hide a real event.
Look at the week of the quietest day in the whole series, 2015-01-27 (the second day of a major
winter storm in New York), across the raw total and both rolling averages. Expect the raw column
to show a sharp one-day drop that the rolling columns barely register, and the 30-day column to
register it even less than the 7-day column.

In [ ]:
pd.DataFrame({"raw": daily, "7-day average": rolling_7, "30-day average": rolling_30}).loc[
    "2015-01-23":"2015-01-30"
]

A wider window trades sensitivity for smoothness. The 30-day average on 2015-01-27 is close to
what it was a week earlier, because one bad day barely moves an average of 30 days. The 7-day
average dips more, because one bad day is a bigger share of 7. The raw number shows the full
drop. None of the three is "wrong", they answer different questions: raw asks what happened today,
a rolling average asks what the recent trend looks like. Activity 6 builds an anomaly detector
that flags exactly this kind of day, and it has to choose its own window with this same trade-off
in mind.

The slider in `activity_5_demand_explorer.py` controls this same `window` argument. Every
value you saw in this notebook, one line per window size, is the same call the app makes each
time a user drags the slider: `series.rolling(window).mean()`.

**Try changing this and re-run, no new code needed:**

1. Change `.rolling(3)` to `.rolling(5)` on the tiny hand-made series. How many leading `NaN`
   values do you get now, and does that match the rule (a window of N needs N values before the
   first real average)?
2. In the plot cell, change the 30-day window to `daily.rolling(60).mean()`. Does the line get
   smoother or rougher, and what does that predict about how well a 60-day average would catch a
   single bad day the way the storm week does?